In [18]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# DLGenAI Project — Milestone 1
## Smart MCQ Solver Challenge — EDA, Text Processing & TF-IDF Baseline

This notebook covers exploratory data analysis, text cleaning, vocabulary
construction, TF-IDF vectorization, cosine-similarity baselines, and MAP@3
evaluation on `train.csv`. Every cell below is executed against the real
dataset — no fine-tuned checkpoints or external downloads are needed for this
milestone, everything runs with `pandas` + `scikit-learn`.



**This milestone has been created to familiarize you with the process of Exploratory Data Analysis (EDA), text processing, and establishing baseline similarity metrics required specifically for Natural Language Processing (NLP) pipelines.** 

**Suggested Readings:**

  1) Tokenization & Text Normalization  

  2) Stop Words & Vocabulary Constraints  

  3) TF-IDF (Term Frequency-Inverse Document Frequency)  

  4) Cosine Similarity in NLP 

  5) Understanding Mean Average Precision (MAP@3)  



## Setup — Imports & Data Loading

In [19]:
import string
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 80)

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print("train shape:", train_df.shape)
train_df.head(10)

train shape: (2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin Heidegger's view on the relati...,Martin Heidegger believes that humans exist within a time continuum that is ...,"Martin Heidegger believes that humans do not exist inside time, but that the...",Martin Heidegger does not believe in the existence of time or that it has an...,Martin Heidegger believes that the relationship between time and human exist...,"Martin Heidegger believes that time is an illusion, and the past, present, a...",B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,Accelerator-based light-ion fusion is a technique that uses particle acceler...,A
2,3,Determine the correct option: What is the term used in astrophysics to descr...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Martin Heidegger's view on the rela...,Martin Heidegger believes that humans exist within a time continuum that is ...,"Martin Heidegger believes that humans do not exist inside time, but that the...",Martin Heidegger does not believe in the existence of time or that it has an...,Martin Heidegger believes that the relationship between time and human exist...,"Martin Heidegger believes that time is an illusion, and the past, present, a...",B
4,5,Identify the correct statement: What is the concept of simultaneity in Einst...,"Simultaneity is relative, meaning that two events that appear simultaneous t...","Simultaneity is relative, meaning that two events that appear simultaneous t...","Simultaneity is absolute, meaning that two events that appear simultaneous t...",Simultaneity is a concept that applies only to Newtonian theories and not to...,Simultaneity is a concept that applies only to relativistic theories and not...,A
5,6,Identify the correct statement: What is the effect generated by a spinning s...,"An electric field, precisely aligned with the spin axis.","A magnetic field, randomly aligned with the spin axis.","A magnetic field, precisely aligned with the spin axis.","A gravitational field, randomly aligned with the spin axis.","A gravitational field, precisely aligned with the spin axis.",C
6,7,"Which of the following is correct? What is a ""coffee ring"" in physics? among...",A type of coffee that is made by boiling coffee grounds in water.,"A pattern left by a particle-laden liquid after it is spilled, named for the...",A type of coffee that is made by mixing instant coffee with hot water.,A type of coffee that is made by pouring hot water over coffee grounds in a ...,"A pattern left by a particle-laden liquid after it evaporates, named for the...",E
7,8,Select the most accurate option: What is the Liouville density? among the li...,The Liouville density is a probability distribution that specifies the proba...,The Liouville density is a quasiprobability distribution that plays an analo...,The Liouville density is a bounded probability distribution that is a conven...,The Liouville density is a probability distribution that takes on negative v...,The Liouville density is a probability distribution that satisfies all the p...,A
8,9,Select the most accurate option: What are the constituents of cold dark matter?,"They are unknown, but possibilities include large objects like MACHOs or new...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.,A
9,10,Which of the following is correct? What is the relationship between the Wign...,"The Wigner function W(x, p) is the Wigner transform of the density matrix op...","The Wigner function W(x, p) is a source function used for the density matrix...","The

## Question 1 — Frequency Distribution of the Correct Answer

Count how often each option (A–E) is the correct answer, then sum the counts of the
most frequent and least frequent options.



In [20]:
answer_counts = train_df["answer"].value_counts()
print("Answer frequency distribution:")
print(answer_counts)

most_frequent_count = answer_counts.max()
least_frequent_count = answer_counts.min()

q1_answer = int(most_frequent_count + least_frequent_count)

print(f"\nMost frequent option : {answer_counts.idxmax()} ({most_frequent_count} occurrences)")
print(f"Least frequent option: {answer_counts.idxmin()} ({least_frequent_count} occurrences)")
print("Q1 answer — sum of most-frequent and least-frequent counts:", q1_answer)


Answer frequency distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most frequent option : B (490 occurrences)
Least frequent option: E (324 occurrences)
Q1 answer — sum of most-frequent and least-frequent counts: 814


## Question 2 — Vocabulary Size After Cleaning `prompt`

Lowercase the `prompt` column, strip all `string.punctuation` characters, split on
whitespace, and count the number of unique words (vocabulary size) across the entire
column.


In [21]:
PUNCT_TABLE = str.maketrans("", "", string.punctuation)

def clean_text(text: str) -> str:
    """Lowercase, strip standard punctuation, collapse to whitespace-split tokens."""
    return str(text).lower().translate(PUNCT_TABLE)

train_df["prompt_clean"] = train_df["prompt"].apply(clean_text)

vocabulary = set()
for cleaned in train_df["prompt_clean"]:
    vocabulary.update(cleaned.split())

q2_answer = len(vocabulary)
print("Q2 answer — vocabulary size across cleaned prompt column:", q2_answer)


Q2 answer — vocabulary size across cleaned prompt column: 859


## Question 3 — Stop-Word Filtering for Row ID 1

Using the cleaned `prompt` for the row with `id == 1`, remove English stop words
(`sklearn.feature_extraction.text.ENGLISH_STOP_WORDS`) and count the remaining words.


In [22]:
row_id1 = train_df.loc[train_df["id"] == 1].iloc[0]
row_id1_words = row_id1["prompt_clean"].split()

row_id1_filtered = [w for w in row_id1_words if w not in ENGLISH_STOP_WORDS]

q3_answer = len(row_id1_filtered)

print("Cleaned prompt (Row ID 1):", row_id1["prompt_clean"])
print("\nWords before filtering:", len(row_id1_words))
print("Words after filtering  :", q3_answer)
print("Remaining words:", row_id1_filtered)


Cleaned prompt (Row ID 1): pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options

Words before filtering: 22
Words after filtering  : 13
Remaining words: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


## Question 4 — TF-IDF Vocabulary Size

Fit a default `TfidfVectorizer(stop_words='english')` on a list containing all the
combined text of the `prompt` and options `A`–`E` across `train.csv` (i.e. every
prompt and every option string treated as one document each), and report the total
number of feature columns (vocabulary size).


In [23]:
OPTION_COLUMNS = ["A", "B", "C", "D", "E"]

corpus = train_df["prompt"].astype(str).tolist()
for col in OPTION_COLUMNS:
    corpus.extend(train_df[col].astype(str).tolist())

print("Total documents in corpus (prompts + all options):", len(corpus))

tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_vectorizer.fit(corpus)

q4_answer = len(tfidf_vectorizer.get_feature_names_out())
print("Q4 answer — TF-IDF vocabulary size:", q4_answer)


Total documents in corpus (prompts + all options): 12000
Q4 answer — TF-IDF vocabulary size: 2762


## Question 5 — Cosine Similarity: Prompt vs Option A (Row ID 1)

Using the TF-IDF vectorizer fitted in Question 4, transform the `prompt` and Option A
for Row ID 1 and compute their cosine similarity.


In [24]:
prompt_vec_id1 = tfidf_vectorizer.transform([row_id1["prompt"]])
option_a_vec_id1 = tfidf_vectorizer.transform([row_id1["A"]])

similarity_prompt_a = cosine_similarity(prompt_vec_id1, option_a_vec_id1)[0, 0]

q5_answer = round(float(similarity_prompt_a), 4)
print("Q5 answer — cosine similarity(prompt, Option A) for Row ID 1:", q5_answer)


Q5 answer — cosine similarity(prompt, Option A) for Row ID 1: 0.2328


## Question 6 — Highest-Similarity Option vs Correct Answer (Full Dataset)

For every row, compute the cosine similarity between the `prompt` and each of the 5
options using the Question 4 vectorizer. Report the percentage of rows where the
option with the highest similarity matches the correct answer.


In [25]:
def most_similar_option(row: pd.Series) -> str:
    prompt_vec = tfidf_vectorizer.transform([row["prompt"]])
    option_vecs = tfidf_vectorizer.transform([row[col] for col in OPTION_COLUMNS])
    sims = cosine_similarity(prompt_vec, option_vecs)[0]
    return OPTION_COLUMNS[int(np.argmax(sims))]

train_df["tfidf_predicted_option"] = train_df.apply(most_similar_option, axis=1)

correct_mask = train_df["tfidf_predicted_option"] == train_df["answer"]
q6_answer = round(100 * correct_mask.mean(), 2)

print("Rows where highest-similarity option matches the correct answer:", correct_mask.sum(), "/", len(train_df))
print("Q6 answer — accuracy percentage:", q6_answer, "%")


Rows where highest-similarity option matches the correct answer: 274 / 2000
Q6 answer — accuracy percentage: 13.7 %


## Question 7 — MAP@3 Example 1

Ground truth = `C`, model predicts `C A B` (in that order). What is the MAP@3 score?


In [26]:
def average_precision_at_3(true_label: str, ranked_predictions: list) -> float:
    """1/rank of the correct answer if within the top 3 predictions, else 0."""
    for rank, pred in enumerate(ranked_predictions[:3], start=1):
        if pred == true_label:
            return 1.0 / rank
    return 0.0

q7_answer = average_precision_at_3("C", ["C", "A", "B"])
print("Q7 answer — MAP@3 (true=C, pred=[C, A, B]):", q7_answer)


Q7 answer — MAP@3 (true=C, pred=[C, A, B]): 1.0


## Question 8 — MAP@3 Example 2

Ground truth = `B`, model predicts `D B E` (in that order). What is the MAP@3 score?


In [27]:
q8_answer = average_precision_at_3("B", ["D", "B", "E"])
print("Q8 answer — MAP@3 (true=B, pred=[D, B, E]):", q8_answer)


Q8 answer — MAP@3 (true=B, pred=[D, B, E]): 0.5


## Question 9 — Majority Class Baseline MAP@3

Rank the answer options by frequency (from Q1). Predict the same static Top-3
`[most_frequent, 2nd_most_frequent, 3rd_most_frequent]` for every row, and compute the
overall MAP@3 on `train.csv`.


In [28]:
ranked_options_by_frequency = answer_counts.index.tolist()  # most frequent -> least frequent
majority_top3 = ranked_options_by_frequency[:3]

print("Options ranked by frequency (desc):", ranked_options_by_frequency)
print("Static Top-3 prediction for every row:", majority_top3)

majority_baseline_scores = train_df["answer"].apply(
    lambda true_label: average_precision_at_3(true_label, majority_top3)
)

q9_answer = round(float(majority_baseline_scores.mean()), 4)
print("Q9 answer — Majority Class Baseline MAP@3:", q9_answer)


Options ranked by frequency (desc): ['B', 'C', 'A', 'D', 'E']
Static Top-3 prediction for every row: ['B', 'C', 'A']
Q9 answer — Majority Class Baseline MAP@3: 0.4213


## Question 10 — TF-IDF Pipeline MAP@3

For every row, compute the TF-IDF cosine similarity between the `prompt` and each of
the 5 options (using the Question 4 vectorizer), sort options from highest to lowest
similarity to form the Top-3 prediction, and compute the average MAP@3 across the
entire training set.


In [29]:
def tfidf_top3(row: pd.Series) -> list:
    prompt_vec = tfidf_vectorizer.transform([row["prompt"]])
    option_vecs = tfidf_vectorizer.transform([row[col] for col in OPTION_COLUMNS])
    sims = cosine_similarity(prompt_vec, option_vecs)[0]
    ranked_idx = np.argsort(sims)[::-1][:3]
    return [OPTION_COLUMNS[i] for i in ranked_idx]

train_df["tfidf_top3"] = train_df.apply(tfidf_top3, axis=1)

tfidf_pipeline_scores = train_df.apply(
    lambda row: average_precision_at_3(row["answer"], row["tfidf_top3"]), axis=1
)

q10_answer = round(float(tfidf_pipeline_scores.mean()), 4)
print("Q10 answer — TF-IDF Pipeline MAP@3 (full train.csv):", q10_answer)


Q10 answer — TF-IDF Pipeline MAP@3 (full train.csv): 0.2762


## Summary of Answers

| Q | Answer |
|---|---|
| Q1 | `{{Q1}}` |
| Q2 | `{{Q2}}` |
| Q3 | `{{Q3}}` |
| Q4 | `{{Q4}}` |
| Q5 | `{{Q5}}` |
| Q6 | `{{Q6}}%` |
| Q7 | `{{Q7}}` |
| Q8 | `{{Q8}}` |
| Q9 | `{{Q9}}` |
| Q10 | `{{Q10}}` |

(This table is filled in automatically below with the real computed values.)


In [30]:
print("Q1  Sum of most/least frequent answer counts :", q1_answer)
print("Q2  Vocabulary size (cleaned prompts)         :", q2_answer)
print("Q3  Words remaining, Row ID 1 after stopwords :", q3_answer)
print("Q4  TF-IDF vocabulary size                    :", q4_answer)
print("Q5  Cosine similarity(prompt, A) Row ID 1     :", q5_answer)
print("Q6  TF-IDF Top-1 accuracy vs correct answer   :", q6_answer, "%")
print("Q7  MAP@3 example 1                           :", q7_answer)
print("Q8  MAP@3 example 2                           :", q8_answer)
print("Q9  Majority Class Baseline MAP@3             :", q9_answer)
print("Q10 TF-IDF Pipeline MAP@3                     :", q10_answer)


Q1  Sum of most/least frequent answer counts : 814
Q2  Vocabulary size (cleaned prompts)         : 859
Q3  Words remaining, Row ID 1 after stopwords : 13
Q4  TF-IDF vocabulary size                    : 2762
Q5  Cosine similarity(prompt, A) Row ID 1     : 0.2328
Q6  TF-IDF Top-1 accuracy vs correct answer   : 13.7 %
Q7  MAP@3 example 1                           : 1.0
Q8  MAP@3 example 2                           : 0.5
Q9  Majority Class Baseline MAP@3             : 0.4213
Q10 TF-IDF Pipeline MAP@3                     : 0.2762
